# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install the `mlcroissant` library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Display the dataset name and description
print(f"Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Version: {metadata.get('version', 'N/A')}")
print(f"License: {metadata.get('license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the record sets, their `@id`s, and the fields available within them. All references use the entity `@id` values per the Croissant schema.

In [ ]:
# Enumerate record sets and their fields using entity @id
ds = dataset
record_sets = ds.metadata.record_sets

print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

# View first records for each record set
for rs in record_sets:
    print(f"Sample record from RecordSet {rs.name} (@id: {rs.id}):")
    for x in ds.records(record_set=rs.id):
        print(x)
        break  # Show only one sample per record set

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
All entities are referenced by their `@id` values.

List all available record sets and choose the main survey record set for further exploration.

In [ ]:
# Collect record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load all records from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for RecordSet @id {record_set_id}:")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))
    print()

# Choose the main survey record set (likely with most fields); pick the record set with the most columns
main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k].columns))
print(f"Selected main RecordSet @id: {main_record_set_id}")

# Display all column @ids for the selected record set
print("Field @ids in the selected RecordSet:")
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            print(f"- {field.name}: {field.id} (type: {field.data_type})")
        break

# Preview the DataFrame from the main record set
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by criteria, normalizing numeric fields, and grouping data by categorical variables.

We will filter and process using fields referenced by their `@id` values.

In [ ]:
# EDA: Filter, normalize, group
main_df = dataframes[main_record_set_id]

# Get numeric fields from schema (using @id)
numeric_fields = []
group_candidates = []
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            # schema data types for integers/floats
            if field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_fields.append(field.id)
            elif field.data_type == 'schema:Text':
                group_candidates.append(field.id)
        break

print(f"Numeric fields (@id): {numeric_fields}")
print(f"Grouping candidates (categorical, @id): {group_candidates}")

# Select a numeric field for analysis (e.g., GAD-7 score)
numeric_field_id = None
for f in numeric_fields:
    if 'gad7' in f.lower():
        numeric_field_id = f
        break
if not numeric_field_id and numeric_fields:
    numeric_field_id = numeric_fields[0]

print(f"Using numeric field @id: {numeric_field_id}")

# Set a threshold, e.g., score > 10 indicates moderate severity
threshold = 10
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, e.g., village or gender
    group_field_id = None
    for f in group_candidates:
        if 'village' in f.lower():
            group_field_id = f
            break
    if not group_field_id and group_candidates:
        group_field_id = group_candidates[0]
    print(f"Grouping using field @id: {group_field_id}")

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} per {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions and relationships using the filtered and grouped results.

Bar plots and histograms display normalized scores and group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized scores
if numeric_field_id and filtered_df is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f"Distribution of normalized scores ({numeric_field_id})")
    plt.xlabel("Normalized Score")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

# Plot mean score by group if grouping was performed
if 'grouped_df' in locals() and group_field_id:
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This exploration loaded a FAIR²-structured mental health survey dataset from Kilifi County, Kenya via its Croissant schema. By referencing entities with their `@id`, we:
- Inspected the available record sets and their fields
- Loaded and previewed survey records via their official IDs
- Performed exploratory filtering and normalization on numeric scores (e.g., GAD-7)
- Grouped average scores by demographic features and visualized the results

This notebook forms a reproducible template for FAIR dataset exploration using `mlcroissant`, enabling scalable and standards-compliant data analysis.